# Code-switch prediction: pipeline (small-scale check on the MacBook)
Flow: **annotate (LID, cached)** -> load / split / examples -> baselines -> XLM-R (-> Qwen) -> save model + threshold -> personalised detection -> demo.
Large-scale runs use `run.py` on HPC (`--model annotate|baselines|xlmr|qwen`); this notebook runs the same code on a few conversations.
Copy `annotations/` from HPC to skip the LID step; `SwitchLingua_text.csv` must sit next to the notebook.

In [1]:
# Set the config BEFORE importing the other codeswitch modules (some defaults bind at import time)
%load_ext autoreload
%autoreload 2
from codeswitch import config as C
C.PAIR_SEL = ["ja-en", "ko-en", "ar-en"]; C.MAX_CONV_PER_PAIR = 40; C.N_TEST_CONV = 20     # tiny
C.EPOCHS = 1; C.QWEN_EPOCHS = 1; C.BATCH_SIZE = 32; C.QWEN_BATCH_SIZE = 8
C.NUM_WORKERS = 1; C.DL_WORKERS = 0                                                # macOS: no worker processes
C.apply_pair_sel()
import json, os
import pandas as pd
from codeswitch import data, lid, metrics, store
from codeswitch.utils import set_seed, DEVICE
set_seed(C.SEED); print(DEVICE, C.PAIR_TAGS)

/Users/WanderingCorn/Documents/Northeastern/CS6140/codeswitch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


mps ['ar-en', 'ja-en', 'ko-en']


## 1. LID: self-test and annotation (cached in `annotations/`)

In [2]:
lid.selftest(C.PAIR_TAGS)          # loads MixLID (and bhasha for hi-en); eyeball the (unit, label) sequences

W1120 23:09:14.254000 95657 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 7647.47it/s]


ar-en [('أنا', 0), ('أحب', 0), ('هذا', 0), ('movie', 1), ('كثيراً', 0), ('because', 1), ("it's", 1), ('great', 1)] | unknown filled: 0
ja-en [('今', 0), ('日', 0), ('は', 0), ('meeting', 1), ('が', 0), ('あ', 0), ('る', 0), ('の', 0), ('で', 0), ("I'll", 1), ('be', 1), ('late', 1)] | unknown filled: 0
ko-en [('나는', 0), ('이', 0), ('movie', 1), ('가', 0), ('정말', 0), ('좋아', 0), ('because', 1), ('it', 1), ('was', 1), ('fun', 1)] | unknown filled: 0


In [3]:
data.annotate_selected(C.DATA_PATH)   # only sentences not in the cache are labelled

[LID ar-en] 186 sentences, 186 cached, 0 to annotate
[LID ja-en] 206 sentences, 206 cached, 0 to annotate
[LID ko-en] 197 sentences, 0 cached, 197 to annotate
  64/197 | 0s
[LID ko-en] done; units with no usable label (filled from neighbour): 0.00%


## 2. Load, split, examples

In [4]:
records = data.load_records(C.DATA_PATH, verbose=False)
data.corpus_report(records)
train_recs, val_recs, test_recs = data.split_records(records)
tr, va, te = (data.build_examples(r) for r in (train_recs, val_recs, test_recs))
for name, r, e in (("train", train_recs, tr), ("val", val_recs, va), ("test", test_recs, te)):
    data.label_stats(r, e, name)

ar-en   sentences    186 | L1 units     1459 | EN units     1122 | sentences with a switch 82.3%
ja-en   sentences    206 | L1 units     5495 | EN units     1590 | sentences with a switch 93.2%
ko-en   sentences    197 | L1 units     1476 | EN units     1242 | sentences with a switch 94.9%
ar-en | أنا فرحانة جداً إن شركة Tecno قامت بتطوير خاصية Multi-Skin Tone اللي بتساعد في تحسين دقة ألوان البشرة في الصور، it's really important for inclusivity.
ar-en | Google عندها Real Tone، وTecno قدمت alternative ممتازة بتعكس اهتمامهم بكل المستخدمين regardless of skin color.
ja-en | 昨日、新しいAppleシリコン搭載Macを見てきたよ, you know?
ja-en | あのエンブレムシール、ちょっと面白いと思った, right?
ko-en | 이번 주말에 친구들이랑 농구하러 갈 거야. And after that, we might go grab some burgers together.
ko-en | 오늘 저녁에 축구 연습이 있을 거야. Then I'll probably finish my homework before bed.
[train] ALL     conv   108 | sent    379 | ex    7919 | switch(0/1) [7130, 789] | dur(0/1/2/3) [7130, 276, 216, 297]
[train] ar-en   conv    36 | sent    123 | ex    1604 | switch

## 3. Baselines

In [5]:
from codeswitch import baselines
base = baselines.run_baselines(tr, va)
metrics.results_table(base, "switch_F1")

,B0 always-switch,B0 never-switch,B1 lookup,B2 char-ngram LR
ALL,0.188,0.0,0.379,0.425
ar-en,0.230,0.0,0.281,0.222
ja-en,0.124,0.0,0.365,0.404
ko-en,0.349,0.0,0.500,0.667


## 4. XLM-R: train, evaluate, save (weights + tokenizer + threshold)

In [6]:
from codeswitch import xlmr
from codeswitch.plots import plot_history
tok, model, tdl, vdl = xlmr.build(tr, va)
hist = xlmr.train(model, tdl, vdl)                     # restores the best epoch
res = xlmr.evaluate(model, vdl)
thr = res["ALL"]["switch_thr"]                         # threshold searched on val, applied unchanged to test
test_res = xlmr.evaluate(model, xlmr.make_eval_dl(te, tok), thr=thr)
metrics.print_scores("test", test_res); print(metrics.pair_report(test_res))
RUN = "out/nb_xlmr"
store.save_run(RUN, "xlmr", model, tok, thr)
json.dump(sorted({r["conv"] for r in test_recs}), open(f"{RUN}/test_convs.json", "w"))
plot_history(hist, f"{RUN}/curves.png")

downsampling: 7919 -> 7919, switch=1 share 10.0%
switch class weights: [0.7452043890953064, 2.240173578262329]
duration class weights (1..3): [0.976165235042572, 1.1034457683563232, 0.941021740436554]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20204.95it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[mem XLM-R step 0] MPS alloc 3.9 GB | driver 6.9 GB | RSS 4.1 GB
epoch 1 step 16/248 (ep 0.065): train loss 1.7787 (sw 0.7531 | dur 1.0256)
  [monitor] SW: PR-AUC 0.198 | Antic-F1 0.206 (max 0.324 @thr 0.535) || DUR: Acc 0.271 | MacroF1 0.142
  ^ best so far
[mem XLM-R step 16] MPS alloc 3.9 GB | driver 4.6 GB | RSS 6.1 GB
epoch 1 step 31/248 (ep 0.125): train loss 1.6787 (sw 0.6407 | dur 1.0380)
  [monitor] SW: PR-AUC 0.408 | Antic-F1 0.000 (max 0.463 @thr 0.415) || DUR: Acc 0.271 | MacroF1 0.142
  ^ best so far
[mem XLM-R step 31] MPS alloc 3.9 GB | driver 4.6 GB | RSS 5.5 GB
epoch 1 step 62/248 (ep 0.250): train loss 1.4678 (sw 0.5262 | dur 0.9416)
  [monitor] SW: PR-AUC 0.652 | Antic-F1 0.021 (max 0.634 @thr 0.369) || DUR: Acc 0.333 | MacroF1 0.267
  ^ best so far
[mem XLM-R step 62] MPS alloc 3.9 GB | driver 4.6 GB | RSS 3.6 GB
epoch 1 step 124/248 (ep 0.500): train loss 1.4648 (sw 0.4160 | dur 1.0487)
  [monitor] SW: PR-AUC 0.640 | Antic-F1 0.577 (max 0.643 @thr 0.591) || DUR: Ac

/Users/WanderingCorn/Documents/Northeastern/CS6140/codeswitch/codeswitch/plots.py:29: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)


## 5. Qwen3 (optional; 0.6B fits on the M1 Pro)

In [7]:
from codeswitch import qwen
qtok, qmodel, qtdl, qvdl, _ = qwen.build(train_recs, val_recs)
qhist = qwen.train(qmodel, qtdl, qvdl)
qres = qwen.evaluate(qmodel, qvdl); qthr = qres["ALL"]["switch_thr"]
qtest = qwen.evaluate(qmodel, qwen.make_eval_dl(test_recs, qtok), thr=qthr)
metrics.print_scores("qwen test", qtest)
store.save_run("out/nb_qwen", "qwen", qmodel, qtok, qthr)
json.dump(sorted({r["conv"] for r in test_recs}), open("out/nb_qwen/test_convs.json", "w"))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


0 / 379 training sentences exceed QWEN_MAX_LEN=256 tokens (tail labels dropped)
switch class weights: [0.7452043890953064, 2.240173578262329]
duration class weights (1..3): [0.976165235042572, 1.1034457683563232, 0.941021740436554]


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 618.36it/s]


KeyboardInterrupt: 

## 6. Personalised detection: type any text, get a prediction at every stopping point

In [8]:
from codeswitch import qwen as _q                      # only for split_chunks (stopping points)
from codeswitch.lid import find_units

def detect(text, predict, model, tok, thr, pair=None):
    """One row per stopping point (after each word / punctuation mark). If pair is given, the LID gold labels of the
    typed text are shown too (loads the LID model)."""
    text = data.clean(text)
    units, stops = _q.split_chunks(text)
    langs = lid.get_tagger(pair).label([text], pair)[0][0] if pair else None
    rows = []
    for end, i in stops:
        if i == len(units) - 1:
            continue                                      # no next unit
        p = predict(model, tok, text[:end], thr=thr)
        row = {"prefix": "..." + text[max(0, end - 14):end], "P(switch)": p["p_switch"], "switch": p["switch"],
               "dur if switch": p["duration_if_switch"], "P(dur 1/2/3)": p["p_duration"]}
        if langs:
            row["gold switch"] = int(langs[i] != langs[i + 1])
        rows.append(row)
    return pd.DataFrame(rows)

detect("我今日去咗 shopping，然后 bought 咗好多嘢", xlmr.predict, model, tok, thr, pair="yue-en")

,prefix,P(switch),switch,dur if switch,P(dur 1/2/3),gold switch
0,...我,0.033,0,1,"[0.519, 0.24, 0.242]",0
1,...我今,0.003,0,1,"[0.704, 0.158, 0.138]",0
2,...我今日,0.034,0,1,"[0.514, 0.286, 0.2]",0
3,...我今日去,0.034,0,1,"[0.5, 0.3, 0.2]",0
4,...我今日去咗,0.086,0,1,"[0.494, 0.214, 0.292]",1
5,...我今日去咗 shopping,0.073,0,1,"[0.498, 0.258, 0.245]",1
6,...今日去咗 shopping，,0.746,0,1,"[0.398, 0.264, 0.338]",1
7,...日去咗 shopping，然,0.044,0,1,"[0.503, 0.277, 0.22]",0
8,...去咗 shopping，然后,0.070,0,1,"[0.462, 0.25, 0.288]",1
9,...ping，然后 bought,0.018,0,1,"[0.471, 0.269, 0.261]",1


## 7. Demo (same as `python -m codeswitch.demo --run out/NAME`)
Shows 10 test conversations (filter by first_language / cs_type), then predicts for any prefix you type. Commands: `:new`, `:filter`, `:q`.

In [9]:
from codeswitch import demo
demo.main(["--run", RUN])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7589.33it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded xlmr model (pairs ['ar-en', 'ja-en', 'ko-en'], threshold 0.8067)

=== Conversation 0 | Korean | cs_type=Intersentential | ko-en#187445 ===
  [0.0] 이번 시즌의 마지막 경기는 정말 숨막히는 승부가 될 거예요.
  [0.1] I will relive every thrilling moment from the Cowboys' rollercoaster game.

=== Conversation 1 | Korean | cs_type=Intrasentential | ko-en#186236 ===
  [1.0] 그 바위는 정말 신기하게 생겼다, almost like it's hiding a secret inside. 외부는 단단한 고체인데, 안에는 액체가 있고, 작은 기체 방울까지 들어있다. 자연이 만들어낸 이 놀라운 구조는, a real masterpiece of geology!

=== Conversation 2 | Japanese | cs_type=Intrasentential | ja-en#182119 ===
  [2.0] ウォン首相は米国の関税政策について交渉の余地がないと話していたけど、this could seriously affect Singapore's economy if global trade tensions keep rising。
  [2.1] シンガポール経済は今すごく不安定で、especially because of the tariff situation between the US and other countries。
  [2.2] 経済の専門家たちは、もし関税が変わらなければ、Singapore might face some big challenges in the near futureと言っている。

=== Conversation 3 | Arabic | cs_type=Intrasentential | ar-en#791 ===
  [3.0] العروض 